# Капстоун · ноутбук 4. Абляции: от гипотезы к измерению

*Курс «Modern LLMs» · итоговый практикум*

Девятнадцать модулей дали нам механизмы и условия их применения. Теперь выберем несколько проверяемых вопросов: как нормализация меняет масштаб активаций, сколько памяти экономит GQA, что меняется при замене FFN экспертами и как модель решает задачу за пределами обучающих длин.

Перед опытом фиксируем изменяемое свойство, остальные настройки и измерение. Иногда изменение требует нескольких согласованных полей конфига: это один содержательный фактор, но не обязательно один флаг. Уравнивание seed делает сравнение воспроизводимым; оно не делает разные архитектуры одинаковыми по всем начальным тензорам.

При первом проходе запустите ячейки и объясните каждую таблицу. Во втором выберите один опыт, повторите его с несколькими seed и проверьте гипотезу. Итоговый отчёт строится из результатов текущего запуска. Быстрый режим проверяет исполнение и не заменяет достаточное обучение.

## 1. Общая установка

In [ ]:
import importlib
import sys
import subprocess
import zipfile
from pathlib import Path

ARCHIVE_GLOB = "modern-llms-capstone-*.zip"

def _activate_capstone_root(candidate: Path):
    package_init = candidate / "minillm" / "__init__.py"
    if not package_init.exists():
        return None
    root = str(candidate.resolve())
    if root not in sys.path:
        sys.path.insert(0, root)
    return root

def _find_capstone_root(search_root: Path):
    direct = _activate_capstone_root(search_root)
    if direct:
        return direct
    for child in sorted(search_root.glob("modern-llms-capstone-*")):
        found = _activate_capstone_root(child)
        if found:
            return found
    return None

def _safe_extract(archive_path: Path, destination: Path):
    destination = destination.resolve()
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            member_path = Path(member.filename)
            if member_path.is_absolute() or ".." in member_path.parts:
                raise RuntimeError(f"unsafe path in archive: {member.filename}")
        archive.extractall(destination)

def ensure_minillm():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        found = _find_capstone_root(candidate)
        if found:
            return "пакет проекта: " + found

    try:
        import minillm  # noqa: F401
        return "minillm уже доступен"
    except ImportError:
        pass

    archives = sorted(Path.cwd().glob(ARCHIVE_GLOB))
    if not archives:
        try:
            from google.colab import files
        except ImportError:
            files = None
        if files is not None:
            print("Загрузите ZIP-пакет Capstone из урока Stepik. Структура каталогов будет восстановлена автоматически.")
            files.upload()
            archives = sorted(Path.cwd().glob(ARCHIVE_GLOB))

    if archives:
        archive_path = archives[-1]
        destination = Path.cwd() / ".capstone_runtime"
        destination.mkdir(exist_ok=True)
        _safe_extract(archive_path, destination)
        matches = sorted(destination.glob("*/minillm/__init__.py"))
        if len(matches) != 1:
            raise RuntimeError(
                f"expected one minillm package in {archive_path.name}, "
                f"found {len(matches)}"
            )
        root = _activate_capstone_root(matches[0].parent.parent)
        if root:
            return "архив проекта: " + str(archive_path)

    raise RuntimeError("minillm не найден. Скачайте языковой ZIP-пакет Capstone из урока Stepik и либо распакуйте его целиком, либо загрузите ZIP в текущую сессию Colab.")

print(ensure_minillm())

for package in ("torch", "numpy", "matplotlib", "regex"):
    try:
        importlib.import_module(package)
    except ImportError:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", package],
            check=True,
        )

import math, os, random
import torch, numpy as np
import matplotlib.pyplot as plt

SEED = 0
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
STEPS = 60 if DEVICE == "cpu" else 600
if os.getenv("CAPSTONE_FAST_VALIDATION") == "1":
    STEPS = 5     # на GPU разница между режимами чётче
print("устройство:", DEVICE, "| шагов на прогон:", STEPS)

In [ ]:
from minillm.config import MiniLLMConfig, kv_bytes_per_token
from minillm.data import TokenStream
from minillm.model import MiniLLM, RMSNorm
from minillm.tokenizer import BBPETokenizer
from minillm.train import TrainConfig, evaluate, train

def high_entropy_corpus(n_lines=8_000, lexicon=600, seed=0) -> str:
    """Тот же псевдоязык, что в ноутбуке 3: зипфовский лексикон оставляет
    моделям запас, в котором различимы архитектурные варианты."""
    rng = random.Random(seed)
    alphabet = "абвгдеиклмнопрстуфхцчшы"
    words = ["".join(rng.choice(alphabet) for _ in range(rng.randint(3, 8)))
             for _ in range(lexicon)]
    weights = [1.0 / (i + 1) ** 0.9 for i in range(lexicon)]
    return "".join(" ".join(rng.choices(words, weights=weights, k=rng.randint(6, 14)))
                   + ".\n" for _ in range(n_lines))

corpus = high_entropy_corpus(seed=SEED)
lines = corpus.splitlines(keepends=True)
a, b = int(len(lines) * .8), int(len(lines) * .9)
train_text, valid_text, test_text = map("".join, (lines[:a], lines[a:b], lines[b:]))
tokenizer = BBPETokenizer.train(train_text, vocab_size=1024, specials=["<|eos|>"])
train_stream = TokenStream.from_text(train_text, tokenizer, 128)
valid_stream = TokenStream.from_text(valid_text, tokenizer, 128)
test_stream = TokenStream.from_text(test_text, tokenizer, 128)
CEILING = math.log(tokenizer.vocab_size)
print(f"словарь {tokenizer.vocab_size} · train {len(train_stream):,} окон · "
      f"valid {len(valid_stream):,} окон · равномерный ориентир ln(V) = {CEILING:.3f}")

In [ ]:
BASE = dict(vocab_size=tokenizer.vocab_size, hidden_size=128, num_hidden_layers=3,
            num_attention_heads=4, num_key_value_heads=2, head_dim=32,
            max_position_embeddings=256)

def run(label: str, steps: int = None, **overrides):
    """Один прогон абляции: фиксированный seed, данные и расписание."""
    cfg = MiniLLMConfig.profile("S", **{**BASE, **overrides})
    torch.manual_seed(SEED)
    model = MiniLLM(cfg).to(DEVICE)
    tcfg = TrainConfig(steps=steps or STEPS, seq_len=128, micro_batch_size=8,
                       optimizer="muon", seed=SEED, device=DEVICE, compute_dtype="auto", log_every=10_000)
    train(model, train_stream, tcfg)
    loss = evaluate(model, valid_stream, tcfg, batches=8)
    total = cfg.parameter_report()["total"]
    print(f"  {label:<26} параметров {total:>9,} · активных "
          f"{cfg.active_parameters():>9,} · valid-лосс {loss:.4f}")
    return dict(label=label, cfg=cfg, model=model, loss=loss, params=total,
                active=cfg.active_parameters())

## 2. Размещение нормы (М6)

Сопоставим pre-norm, post-norm, Peri-LN, схему OLMo2 и контроль без нормализации. Они нормируют разные участки вычислительного графа; одинаковое название масштаба активаций не делает их эквивалентными.

Начнём с отдельной остаточной цепочки $h\leftarrow h+\tanh(h)W$ со случайными матрицами, элементы которых имеют дисперсию $1/d$. Это нелинейная иллюстрация, а не точная линейная рекурсия лекции. Проследим RMS по 36 слоям и сравним расположение нормы. Ограниченность tanh меняет характер роста, поэтому заранее обещать экспоненту нельзя.

In [ ]:
torch.manual_seed(1)
d, layers = 256, 36
h0 = torch.randn(1, d)
weights = [torch.randn(d, d) / math.sqrt(d) for _ in range(layers)]
norm = RMSNorm(d, 1e-6)

traces = {}
with torch.no_grad():
    for mode in ("none", "pre", "post"):
        h, trace = h0.clone(), []
        for W in weights:
            if mode == "none":
                h = h + torch.tanh(h) @ W
            elif mode == "pre":
                h = h + torch.tanh(norm(h)) @ W
            else:
                h = norm(h + torch.tanh(h) @ W)
            trace.append(h.pow(2).mean().sqrt().item())
        traces[mode] = trace

plt.figure(figsize=(7, 4))
for mode, color in (("none", "#BF616A"), ("pre", "#5E81AC"), ("post", "#A3BE8C")):
    plt.plot(range(1, layers + 1), traces[mode], label=mode, color=color)
plt.yscale("log"); plt.xlabel("слой"); plt.ylabel("rms потока")
plt.title("Нелинейная остаточная цепочка: 36 слоёв")
plt.legend(); plt.gca().spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

for mode, trace in traces.items():
    print(f"{mode:>5}: слой 1 = {trace[0]:8.3f} → слой 36 = {trace[-1]:10.3f}")

Сравните начальный и конечный RMS каждой ветви. Нормализация после сложения удерживает RMS около единицы с поправкой на epsilon; две другие ветви могут накапливать обновления. Масштаб наблюдаемого роста нужно читать по графику.

Следующий опыт переносит вопрос на обучаемую MiniLLM с тремя слоями. Здесь меняются инициализация и форма подслоя, поэтому численный результат предыдущей цепочки не является прогнозом для модели. Сопоставим loss и профиль RMS, сохранив бюджет и настройки оптимизатора.

In [ ]:
def stream_rms(model: MiniLLM) -> list[float]:
    ids = valid_stream.batch(torch.arange(4)).to(DEVICE)
    values = []
    with torch.no_grad():
        x = model.embed_tokens(ids)
        cos, sin = model.rope(torch.arange(ids.shape[1], device=DEVICE))
        for layer in model.layers:
            x = layer(x, cos, sin)
            values.append(x.float().pow(2).mean().sqrt().item())
    return values

print("абляция нормы:")
norm_runs = [run(f"norm_placement={p}", norm_placement=p)
             for p in ("pre", "post", "peri", "olmo2", "none")]

print("\nrms residual-потока по слоям обученных моделей:")
for r in norm_runs:
    r["rms"] = stream_rms(r["model"])
    print(f"  {r['label']:<26} " +
          " → ".join(f"{v:.2f}" for v in r["rms"]))

Низкий loss и умеренный RMS отвечают на разные вопросы. Если вариант проигрывает по loss, сначала посмотрите на масштаб его активаций и градиентов: из одного ухудшения нельзя заключить, что остаточный поток «взорвался». Равным образом небольшой рост на трёх слоях не опровергает проблему на большей глубине.

Peri-LN нормирует обе стороны подслоя, а схема OLMo2 — выход перед сложением. Это меняет эффективный масштаб обновления, поэтому общий learning rate полезен как контроль, но не как окончательный подбор для всех вариантов. Таблица характеризует заданные настройки. Для сравнения методов потребуется настройка каждой схемы и несколько независимых запусков.

## 3. MHA / GQA / MQA (M5)

Здесь масштаб не мешает: экономия KV-кэша — свойство геометрии, а не обучения,
и считается точно. Открытый вопрос другой: что модель теряет в качестве.

In [ ]:
print("абляция внимания:")
attn_runs = []
for name, kv in (("MHA (H_kv=4)", 4), ("GQA (H_kv=2)", 2), ("MQA (H_kv=1)", 1)):
    r = run(name, num_key_value_heads=kv)
    r["kv_per_token"] = kv_bytes_per_token(r["cfg"])
    attn_runs.append(r)

print(f"\n{'схема':<14} {'KV, Б/токен':>12} {'×к MHA':>8} {'valid-лосс':>11} {'Δлосс':>8}")
mha = attn_runs[0]
for r in attn_runs:
    print(f"{r['label']:<14} {r['kv_per_token']:>12,} "
          f"{mha['kv_per_token'] / r['kv_per_token']:>7.1f}× "
          f"{r['loss']:>11.4f} {r['loss'] - mha['loss']:>+8.4f}")

Отношение объёмов KV равно `H/H_kv` при общей длине, числе слоёв и типе данных. Здесь оно выводится из конфигурации и не зависит от качества обучения. Разность loss — уже эмпирическое наблюдение: сравните её знак и величину в текущем запуске.

Из небольшого расхождения нельзя заключить, что сокращение числа KV-голов всегда бесплатно. Из большого — что один вариант внимания хуже при любом бюджете. Изменились также число параметров проекций и ход оптимизации; для более общего вывода нужны повторности и отдельный подбор настроек.

## 4. Активации внутри GLU (М6)

Сравним SwiGLU, GeGLU и ReGLU. Все три используют две входные матрицы и одну выходную; отличается функция в ветви гейта. При одинаковой ширине число параметров совпадает, что проверит assert.

Правило $d_{ff}\approx8d/3$ связывает бюджет трёхматричного GLU с классическим двухматричным FFN ширины $4d$. Здесь ширина дополнительно округляется, а сам двухматричный вариант не реализован. Поэтому опыт отвечает на вопрос о выборе активации внутри GLU, а не о пользе гейта вообще.

In [ ]:
d = BASE["hidden_size"]
gated_ff = round(8 * d / 3 / 64) * 64          # правило 2/3, выровнено под 64
print(f"все варианты: 3 · {d} · {gated_ff} = {3 * d * gated_ff:,} параметров FFN\n")

print("абляция φ внутри GLU-семейства:")
ffn_runs = [run(f"{name} (3 матрицы)", hidden_act=act, intermediate_size=gated_ff)
            for name, act in (("SwiGLU", "swiglu"), ("GeGLU", "geglu"),
                              ("ReGLU", "relu"))]

assert len({r["params"] for r in ffn_runs}) == 1, "бюджеты обязаны совпадать"
best = min(ffn_runs, key=lambda r: r["loss"])
spread = max(r["loss"] for r in ffn_runs) - min(r["loss"] for r in ffn_runs)
print(f"\nбюджеты совпадают до параметра: {ffn_runs[0]['params']:,}")
print(f"лучший на этом прогоне — {best['label']}, разброс по лоссу {spread:.4f}")

Проверьте совпадение бюджетов и размах loss. Код назовёт минимальный loss данного запуска, но один seed не позволяет оценить разброс между запусками. Чтобы назвать различие устойчивым, повторите сравнение с несколькими seed, сохраняя общий протокол и одинаковую проверочную выборку.

Параметр `hidden_act="relu"` здесь означает ReGLU: наличие ReLU не убирает вторую входную матрицу. Это пример того, зачем читать и конфигурацию, и вычислительный граф. Сравнение с обычным FFN осталось за пределами данного опыта.

## 5. Плотный FFN и MoE при близком активном бюджете (М7)

Сохраним словарь, внимание, глубину и позиции, а плотный FFN заменим экспертами. Ширина эксперта округляется до кратного 64, поэтому активные параметры могут различаться. Таблица покажет их отношение вместе с общим числом параметров и loss; KV-геометрия останется общей.

Обе ветви получают удвоенный относительно остальных опытов бюджет шагов. Это уравнивает число обработанных токенов, но не время и не все FLOP. Маршрутизация и объединение ответов экспертов тоже стоят вычислений. Цель — наблюдать заданную пару моделей и отдельно исследовать нагрузку роутера.

In [ ]:
from minillm.model import MoELayer

MOE_STEPS = STEPS * 2
EXPERTS, TOPK, SHARED = 8, 2, 1
expert_ff = max(64, round(gated_ff / (TOPK + SHARED) / 64) * 64)

print("абляция разреженности (бюджет шагов удвоен):")
dense = run("dense", steps=MOE_STEPS, intermediate_size=gated_ff)
moe = run(f"MoE {EXPERTS} экспертов, top-{TOPK}", steps=MOE_STEPS,
          intermediate_size=gated_ff, num_local_experts=EXPERTS,
          num_experts_per_tok=TOPK, num_shared_experts=SHARED,
          moe_intermediate_size=expert_ff, first_k_dense_replace=1)

parity = moe["active"] / dense["active"]
sparsity = moe["params"] / moe["active"]
print(f"\n{'вариант':<26} {'total':>12} {'active':>11} {'valid-лосс':>11}")
for r in (dense, moe):
    print(f"{r['label']:<26} {r['params']:>12,} {r['active']:>11,} {r['loss']:>11.4f}")
print(f"\nактивных у MoE к плотному: ×{parity:.3f} — активные параметры; время измеряется отдельно")
print(f"ёмкость: total ×{moe['params'] / dense['params']:.2f}, "
      f"разрежённость ×{sparsity:.2f}")
print(f"KV-кэш совпадает до байта: "
      f"{kv_bytes_per_token(dense['cfg']) == kv_bytes_per_token(moe['cfg'])}")

In [ ]:
# Что с нагрузкой экспертов? Это первая вещь, которую надо смотреть у MoE.
def load_table(result, title):
    layers = [l.ffn for l in result["model"].layers if isinstance(l.ffn, MoELayer)]
    print(title)
    print(f"{'слой':>5} {'min':>7} {'max':>7} {'std':>7}  {'max|bias|':>9}")
    for i, layer in enumerate(layers, start=1):
        load = layer.expert_load
        print(f"{i:>5} {load.min():>7.3f} {load.max():>7.3f} {load.std():>7.3f}"
              f"  {layer.expert_bias.abs().max():>9.4f}")
    return max(float(l.expert_load.std()) for l in layers)

ideal = TOPK / EXPERTS
print(f"идеально равномерная нагрузка — {ideal:.3f} на эксперта\n")
std_default = load_table(moe, "смещение по умолчанию (router_bias_speed = 1e-3):")

Посмотрите на минимум, максимум и стандартное отклонение нагрузки. Это снимок последнего измеренного батча, а не среднее по всему обучению. Сильная концентрация назначений может указывать на недостаточный баланс; один снимок не устанавливает устойчивый коллапс.

Теперь увеличим скорость обновления балансирующего смещения с $10^{-3}$ до $5\cdot10^{-2}$, сохранив остальные настройки. Сравним и нагрузку, и loss: уменьшение разброса не обязано улучшить предсказания, поскольку роутер одновременно учится выбирать полезных экспертов.

In [ ]:
moe_fast = run(f"MoE, balance ×50", steps=MOE_STEPS, intermediate_size=gated_ff,
               num_local_experts=EXPERTS, num_experts_per_tok=TOPK,
               num_shared_experts=SHARED, moe_intermediate_size=expert_ff,
               first_k_dense_replace=1, router_bias_speed=5e-2)
print()
std_fast = load_table(moe_fast, "ускоренное смещение (router_bias_speed = 5e-2):")
print(f"\nмаксимальный разброс нагрузки: {std_default:.3f} → {std_fast:.3f}")
print(f"valid-лосс: {moe['loss']:.4f} → {moe_fast['loss']:.4f} "
      f"(dense {dense['loss']:.4f})")

### Ёмкость эксперта: сколько теряется молча

Буфер эксперта конечен, а роутер об этом не знает. `capacity_factor` задаёт
запас над идеально равномерной нагрузкой; всё, что не влезло, отбрасывается.
Прогон ниже — только forward на обученной модели, обучение не трогаем.

In [ ]:
probe = valid_stream.batch(torch.arange(8)).to(DEVICE)
print(f"{'capacity_factor':>16} {'потолок':>9} {'отброшено':>11}")
capacity_rows = []
for factor in (None, 2.0, 1.5, 1.25, 1.0, 0.75):
    cfg_c = MiniLLMConfig(**{**moe["cfg"].to_dict(), "capacity_factor": factor})
    probe_model = MiniLLM(cfg_c).to(DEVICE)
    probe_model.load_state_dict(moe["model"].state_dict())
    with torch.no_grad():
        probe_model(probe)
    layer = next(l.ffn for l in probe_model.layers if isinstance(l.ffn, MoELayer))
    cap = cfg_c.expert_capacity(probe.numel())
    capacity_rows.append(dict(factor=factor, capacity=cap, dropped=float(layer.dropped_fraction)))
    print(f"{str(factor):>16} {str(cap):>9} {layer.dropped_fraction.item():>10.1%}")

Числа total/active и равенство KV следуют из конфигураций; loss и нагрузка получены в опыте. Сопоставляйте эти виды свидетельств отдельно. Если ускоренная балансировка уменьшила разброс, но разрыв с плотной моделью остался, этого недостаточно, чтобы исключить дисбаланс из причин отставания: вмешательство могло быть частичным и само изменить оптимизацию.

При равномерных назначениях top-2 из восьми даёт каждому эксперту в среднем четверть токенов. Это условное среднее, а не гарантированная нагрузка и не число шагов оптимизатора. Короткий бюджет ограничивает опыт, но не объясняет автоматически любой результат.

Таблица capacity показывает долю отброшенных назначений при одной и той же модели. Предел равен запасу над равномерной нагрузкой, поэтому при неравномерном распределении возможны потери. Отбрасывание назначения не удаляет весь токен: остаются остаточная ветвь, внимание и другие эксперты. Для оценки последствий дополнительно измерьте loss под каждым ограничением; текущая таблица измеряет именно отбрасывание.

## 6. RoPE за пределами обучающих длин (М4)

Для исследования дальней зависимости нужна задача, в которой она существует. Поэтому отдельно обучим модель на случайной последовательности, повторённой дважды. Цели во второй половине можно восстановить по первой; длина половины задаёт расстояние зависимости. Успех такого копирования не доказывает наличие конкретной induction head — для этого потребовался бы анализ внутренних вычислений.

Обучающие расстояния лежат в диапазоне 12–48, проверочные — 48–128. Сначала измерим точность на расстоянии 32: если модель ещё не освоила задачу внутри диапазона, падение вне его нельзя уверенно назвать срывом экстраполяции. Равномерное случайное угадывание даёт $1/64=0{,}015625$. Быстрый режим делает лишь десять обучающих шагов и служит проверкой исполнения.


Длина последовательности меняется от батча к батчу, поэтому бюджет считаем по фактическим входным позициям и обучаемым целям. Значение `seq_len` в конфигурации обучения не задаёт длину этого генератора. Следующий код сохраняет оба счётчика для каждого варианта RoPE.

In [ ]:
from minillm.probes import RepeatTask, chance_level, repeat_accuracy
from minillm.model import RotaryEmbedding

PROBE_V = 64
probe = RepeatTask(vocab_size=PROBE_V, train_range=(12, 48), seed=SEED)
PROBE_BASE = dict(vocab_size=PROBE_V, hidden_size=128, num_hidden_layers=2,
                  num_attention_heads=4, num_key_value_heads=2, head_dim=32,
                  max_position_embeddings=512)
PROBE_STEPS = 1200 if DEVICE == "cpu" else 3000
if os.getenv("CAPSTONE_FAST_VALIDATION") == "1":
    PROBE_STEPS = 10

probe_budgets = {}

def train_probe(theta: float) -> MiniLLM:
    cfg = MiniLLMConfig.profile("S", **{**PROBE_BASE, "rope_theta": theta})
    torch.manual_seed(SEED)
    model = MiniLLM(cfg).to(DEVICE)
    probe_state = train(model, probe, TrainConfig(steps=PROBE_STEPS, seq_len=64,
                                    micro_batch_size=16, optimizer="muon",
                                    seed=SEED, device=DEVICE, log_every=10_000))
    probe_budgets[str(theta)] = {
        "input_positions": probe_state.tokens,
        "supervised_targets": probe_state.supervised_tokens,
    }
    return model

probe_models = {theta: train_probe(theta) for theta in (10_000.0, 500_000.0)}
print(f"уровень случая: {chance_level(probe):.4f}")
print("измеренный бюджет RoPE:", probe_budgets)
probe_in_range = {str(theta): repeat_accuracy(model, probe, 32, device=DEVICE)
                  for theta, model in probe_models.items()}
for theta, accuracy in probe_in_range.items():
    print("rope_theta:", theta, "in-range accuracy at 32:", accuracy)

In [ ]:
LENGTHS = (48, 64, 80, 96, 128)
rope_rows = {f"base {theta:g}": [repeat_accuracy(m, probe, n, device=DEVICE)
                                 for n in LENGTHS]
             for theta, m in probe_models.items()}

# YaRN поверх уже обученной модели: у RoPE нет параметров, поэтому достаточно
# заменить таблицу частот — веса остаются те же. Это и есть «растяжка без
# дообучения» из M4. Окно претрейна равно 96 токенам (две половины по 48).
for factor in (2.0, 4.0):
    model = probe_models[10_000.0]
    saved = model.rope
    yarn_cfg = MiniLLMConfig.profile("S", **{**PROBE_BASE, "rope_theta": 10_000.0,
        "rope_scaling": {"rope_type": "yarn", "factor": factor,
                         "original_max_position_embeddings": 96}})
    model.rope = RotaryEmbedding(yarn_cfg).to(DEVICE)
    rope_rows[f"10000 + YaRN ×{factor:g}"] = [
        repeat_accuracy(model, probe, n, device=DEVICE) for n in LENGTHS]
    model.rope = saved

print(f"{'вариант':<22}" + " ".join(f"{n:>8}" for n in LENGTHS))
print("-" * 64)
for name, values in rope_rows.items():
    print(f"{name:<22}" + " ".join(f"{v:>8.3f}" for v in values))
print("\nобучение шло на расстояниях 12–48; всё правее 48 — экстраполяция")

In [ ]:
plt.figure(figsize=(7.5, 4.5))
palette = ("#BF616A", "#D08770", "#5E81AC", "#A3BE8C")
for (name, values), color in zip(rope_rows.items(), palette):
    plt.plot(LENGTHS, values, "o-", label=name, color=color)
plt.axvline(48, ls="--", lw=1, color="#4C566A")
plt.text(49, 0.5, "граница обучения", rotation=90, va="center",
         fontsize=9, color="#4C566A")
plt.axhline(chance_level(probe), ls=":", lw=1, color="#4C566A")
plt.text(120, chance_level(probe) + 0.03, "случайное угадывание",
         ha="right", fontsize=9, color="#4C566A")
plt.xlabel("расстояние зависимости (длина половины)")
plt.ylabel("доля верно восстановленных токенов")
plt.title('Точность по расстоянию зависимости')
plt.legend(); plt.ylim(-0.03, 1.05)
plt.gca().spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

Прочитайте график в два шага. Сначала сравните обе базы RoPE внутри обучающего диапазона и за его пределами. Затем для модели с базой 10 000 сопоставьте исходную таблицу частот с YaRN при неизменных весах. При замене RoPE используется реализация YaRN из пакета; параметры внимания, предусмотренные этим механизмом, тоже входят в вмешательство.

Улучшение на большой длине и ухудшение на малой, если они наблюдаются, относятся к этому обучению и выбранному фактору. Ни полный провал обычного RoPE, ни восстановление качества YaRN заранее не гарантированы. В отчёте сохраняются все точки, включая точность внутри диапазона; выбор лучшего фактора по этим точкам уже становится настройкой и требует нового теста.

## 7. Muon и AdamW: подбор и независимая проверка

Сравнение двух значений по умолчанию показывает две настройки, а не возможности оптимизаторов в целом. Здесь для каждого перебираем две learning rate при общем бюджете. Выбираем минимум по validation, а выбранную модель оцениваем на отдельном test, который не использовался при выборе.

Разделение сделано до обучения токенизатора. Все части созданы одним генератором, поэтому test проверяет перенос на новые синтетические строки, а не на естественный язык. Изменение seed и расширение сетки остаются отдельным следующим опытом.

In [ ]:
def run_optimizer(optimizer: str, lr: float):
    cfg = MiniLLMConfig.profile("S", **BASE)
    torch.manual_seed(SEED)
    model = MiniLLM(cfg).to(DEVICE)
    tcfg = TrainConfig(steps=STEPS, seq_len=128, micro_batch_size=8,
                       optimizer=optimizer, lr=lr if optimizer == "adamw" else 3e-3,
                       muon_lr=lr if optimizer == "muon" else 0.02,
                       seed=SEED, device=DEVICE, log_every=10_000)
    train(model, train_stream, tcfg)
    return dict(model=model, config=tcfg, valid_loss=evaluate(model, valid_stream, tcfg, batches=8))

grid = {
    "adamw": [3e-3, 1e-2],
    "muon":  [2e-2, 6e-2],
}
optimizer_report = {}
for optimizer, lrs in grid.items():
    runs = {lr: run_optimizer(optimizer, lr) for lr in lrs}
    chosen_lr, selected = min(runs.items(), key=lambda kv: kv[1]["valid_loss"])
    test_loss = evaluate(selected["model"], test_stream, selected["config"], batches=8)
    optimizer_report[optimizer] = dict(selected_lr=chosen_lr,
        validation_grid={str(lr): r["valid_loss"] for lr,r in runs.items()},
        selected_valid_loss=selected["valid_loss"], selected_test_loss=test_loss)
    print(optimizer, optimizer_report[optimizer])

Сравните выбранные learning rate, validation loss и test loss. Разность на test не участвует в выборе и потому даёт более честную проверку данного решения. Две точки сетки не гарантируют нахождения лучших настроек; остальные коэффициенты оптимизаторов остаются фиксированными.

Число шагов напечатано в начале: по умолчанию 60 на CPU или 600 на GPU, в быстром режиме — 5. Короткий горизонт может благоприятствовать быстрому старту и не предсказывает итог длинного обучения. Один seed не позволяет оценить устойчивость преимущества: для этого повторите весь цикл подбора и проверки.

## 8. От измерений к сохранённому отчёту

Сохраним настройки и численные результаты в JSON, прочитаем файл обратно и построим сводку. Так таблица связана с фактическим запуском и может быть проверена отдельно от обучающих ячеек. Расчётные величины будут помечены как расчётные, а loss, RMS, нагрузка и точность — как наблюдения.

Часть вопросов курса здесь не проверяется: например, ускорение упакованной квантизации или качество RL. Для них не будет ни выдуманного числа, ни вердикта о воспроизведении. В быстром режиме статус отчёта явно указывает на проверку исполнения.

In [ ]:
CAPSTONE_LANGUAGE = 'ru'
"""Embedded into both ablation notebooks; reads only this run's measurements."""
import json
from pathlib import Path

ablation_record = {
    "schema": 2,
    "language": CAPSTONE_LANGUAGE,
    "mode": "fast_execution_check" if os.getenv("CAPSTONE_FAST_VALIDATION") == "1" else "teaching_experiment",
    "seed": SEED, "device": DEVICE, "torch": torch.__version__,
    "steps": STEPS, "moe_steps": MOE_STEPS, "probe_steps": PROBE_STEPS,
    "data": {"train_sha": train_stream.checksum(), "valid_sha": valid_stream.checksum(),
             "test_sha": test_stream.checksum()},
    "chain_rms": traces,
    "norm": [{"label": r["label"], "valid_loss": r["loss"], "rms": r["rms"]} for r in norm_runs],
    "attention": [{"label": r["label"], "valid_loss": r["loss"],
                   "calculated_kv_bytes_per_token_bf16": r["kv_per_token"]} for r in attn_runs],
    "glu": [{"label": r["label"], "valid_loss": r["loss"], "parameters": r["params"]} for r in ffn_runs],
    "moe": {"dense_total": dense["params"], "moe_total": moe["params"],
            "dense_active": dense["active"], "moe_active": moe["active"],
            "active_ratio": parity, "dense_loss": dense["loss"], "moe_loss": moe["loss"],
            "faster_balance_loss": moe_fast["loss"],
            "last_batch_load_std": std_default, "faster_balance_load_std": std_fast},
    "capacity": capacity_rows,
    "rope": {"distances": list(LENGTHS), "accuracy": rope_rows,
             "training_budgets": probe_budgets,
             "in_range_32": probe_in_range, "chance": chance_level(probe)},
    "optimizers": optimizer_report,
    "not_tested": ["ungated FFN", "packed quantization speed", "RL answer quality",
                   "scaling-law optimum", "QK-norm ablation",
                   "speculative distribution test in this notebook", "EOT ablation in this notebook"],
}
report_dir = Path("runs/ablations")
report_dir.mkdir(parents=True, exist_ok=True)
report_path = report_dir / "measurements.json"
report_path.write_text(json.dumps(ablation_record, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
saved = json.loads(report_path.read_text(encoding="utf-8"))
assert saved == ablation_record
ru = CAPSTONE_LANGUAGE == "ru"
mode_ru = {"fast_execution_check": "быстрая проверка исполнения", "teaching_experiment": "учебный эксперимент"}
print("Режим:" if ru else "Mode:", mode_ru[saved["mode"]] if ru else saved["mode"])
print("Измерения сохранены:" if ru else "Measurements saved:", report_path)
labels_ru = {"chain_rms": "RMS остаточной цепочки", "norm": "Нормализация",
             "attention": "Внимание", "glu": "Варианты GLU", "moe": "Эксперты",
             "capacity": "Ограничение нагрузки", "rope": "Позиции", "optimizers": "Оптимизаторы"}
for key in ("chain_rms", "norm", "attention", "glu", "moe", "capacity", "rope", "optimizers"):
    print("\n" + (labels_ru[key] if ru else key))
    print(json.dumps(saved[key], ensure_ascii=False, indent=2))
print("\nЗдесь не проверялось:" if ru else "\nNot tested here:")
not_tested_ru = {"ungated FFN": "FFN без гейта",
    "packed quantization speed": "скорость упакованной квантизации",
    "RL answer quality": "качество ответов после RL",
    "scaling-law optimum": "оптимум закона масштабирования",
    "QK-norm ablation": "абляция QK-нормировки",
    "speculative distribution test in this notebook": "распределение спекулятивного семплирования в этой тетради",
    "EOT ablation in this notebook": "абляция EOT в этой тетради"}
for item in saved["not_tested"]:
    print("-", not_tested_ru[item] if ru else item)

## 9. Как продолжить исследование

Выберите одно наблюдение из отчёта и сформулируйте следующий вопрос так, чтобы новый опыт мог изменить ваше мнение. Например, сохранится ли разность loss после подбора learning rate для каждой нормы? Изменится ли MoE при другом распределении данных? Освоена ли задача повторения до того, как вы оцениваете экстраполяцию?

В каждом случае сохраните контроль, бюджет, данные и правило выбора. Если результат зависит от seed, покажите разброс; если измерение не проводилось, оставьте его отсутствующим. Отсутствие заметного эффекта в одном опыте ограничивает вывод об этом опыте и не отменяет механизм вообще.

Итоговый практикум завершает курс умением связывать формулу, код и свидетельство. Самостоятельный отчёт должен позволить другому читателю понять, что измерено, что вычислено из конфигурации и какое более сильное утверждение ещё требует проверки.